# AI 智能应用综合实践：课程选课智能助手

本 Notebook 是项目的**学习与开发框架**，不是完成后的答案。请结合 `docs/项目任务说明.docx`，在 `course_engine.py`、`course_mcp_server.py`、`web_search.py` 中完成各项任务，再通过网页测试。

建议使用项目虚拟环境作为 Notebook 内核。模型、Prompt、RAG、工具、MCP、Skill、多 Agent 和 LangGraph 的具体实现均需自行完成。

## 1. 确认项目目录和演示资源

本单元只定位文件并检查资源是否存在，不调用模型，也不修改数据库。

In [ ]:
from pathlib import Path
import sys

# 兼容从项目目录打开 Notebook，或从其父目录启动内核。
ROOT = Path.cwd()
if not (ROOT / 'course_engine.py').exists() and (ROOT / '综合大项目' / 'course_engine.py').exists():
    ROOT = ROOT / '综合大项目'
if not (ROOT / 'course_engine.py').exists():
    raise RuntimeError('请将 Notebook 工作目录切换到“综合大项目”或其父目录。')
ROOT = ROOT.resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('项目目录：', ROOT)
print('解释器：', sys.executable)
print('数据库存在：', (ROOT / 'data' / 'course_selection.sqlite3').is_file())
print('课程文档：', [p.name for p in sorted((ROOT / 'knowledge' / 'courses').glob('*.md'))])

## 2. 分析需求与数据库

1. 明确课程咨询、比较推荐、教室和名额查询、选课、退课及多轮追问。
2. 设计每门课最多 20 人、每人最多选 2 门、禁止重复选课等规则。
3. 查看 `users`、`courses`、`enrollments` 三张表，说明主键、外键和唯一约束。
4. C004 已预置满员，其余课程初始没有选课记录；操作数据库前可复制备份。

In [ ]:
import sqlite3

# 只读取表结构；业务查询和写入工具需要自行开发。
db_path = ROOT / 'data' / 'course_selection.sqlite3'
with sqlite3.connect(f'file:{db_path}?mode=ro', uri=True) as connection:
    for table_name in ('users', 'courses', 'enrollments'):
        print(table_name, [item[1] for item in connection.execute(f'PRAGMA table_info({table_name})')])

## 3. Model 和 Prompt

在 `course_engine.py` 中完成 `create_model()` 与 `build_course_agent()`：

- 从 `../../.env` 读取配置，使用 DeepSeek Flash，`temperature=0.5`。
- 设计角色、任务、工具使用、回答范围和输出方式。
- 选同一个问题比较提示词优化前后的输出。不要在 Notebook 或报告中输出密钥。

In [ ]:
# TODO：完成 course_engine.py 的模型和 Prompt，再在此记录自己的测试。
# 提示：本单元只提供位置，具体调用方式由你实现。

## 4. 课程知识库和 RAG

- 加载 `knowledge/courses` 中的课程总览和五门课程文档。
- 按章节切分，使用通义千问 `text-embedding-v1` 向量化并建立检索。
- 对课程咨询、追问和比较进行检索；资料不足时说明未找到依据。
- 在 `course_engine.py` 完成 `build_retriever()`，并在报告中记录资料与检索结果。

In [ ]:
# TODO：实现并测试知识库加载、向量化和检索。
# 验证 C001、C002 的比较，以及知识库没有答案的问题。

## 5. 课程业务工具与 MCP Server

在 `course_mcp_server.py` 创建并启动 MCP Server，注册 `list_courses`、`list_enrollments`、`select_course`、`drop_course`。自行实现 SQL 和异常处理，再在 `course_engine.py` 中连接服务。

选课和退课写入前必须确认；数据库读写时自行保护选课上限、课程容量及并发场景。

In [ ]:
# TODO：实现 MCP Server 和客户端。
# 分别测试课程目录、我的选课、选课、退课、已满及重复选择。

## 6. Tavily 搜索 MCP

在 `web_search.py` 中完成 Tavily MCP 的配置与调用，从 `.env` 读取 `TAVILY_API_KEY`。按需把网络资料与课程知识结合，展示真实来源；服务不可用时应给出提示并继续使用本地知识。

In [ ]:
# TODO：实现并测试 Tavily 搜索、来源返回与异常回退。

## 7. LangGraph 工作流和人工确认

在 `course_engine.py` 定义状态、节点和条件路由，整合课程咨询、目录、我的选课、选课及退课。选退课操作在执行前等待用户确认，取消时不写数据库；注意同一会话的上下文和状态。

In [ ]:
# TODO：实现工作流，绘制节点图，并测试确认、取消、满员和多轮追问。

## 8. Skill 和多 Agent

设计课程推荐 Skill，写清触发条件、输入和输出；划分课程咨询 Agent 与选课业务 Agent 的职责，记录两者交接数据与联调结果。

In [ ]:
# TODO：实现 Skill 与多 Agent 协作，并记录自己的测试。

## 9. 连接网页与完成验收

完成 `course_engine.chat()`，返回 `reply`、`waiting_for_confirmation`、`pending_action`，使网页能够展示回复和确认按钮。启动 `python app.py` 后在浏览器访问 `http://127.0.0.1:8765`。

至少覆盖课程咨询、追问、课程比较、满员、个人上限、取消与确认、选课后查库、退课后名额、Tavily 异常和重启后数据库记录。按报告模板填写真实截图、测试结果和总结。

In [ ]:
# TODO：按任务说明执行测试；不得把未完成的功能写成已通过。